# 02 - Diagnóstico de ratings

Inspecciona Elo, PI rating, rolling-form y el rating compuesto. Sirve para validar que los ratings tengan sentido antes de entrenar modelos.

**Prerequisito**: tabla canónica poblada (`data/interim/matches_unified.csv`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.data_loader import DataLoader
from src.ratings.elo import EloRating
from src.ratings.form_rating import FormRating
from src.ratings.pi_rating import PIRating
from src.ratings.rating_ensemble import RatingEnsemble

## Ajuste de los tres rating systems

In [ ]:
matches = DataLoader().load_matches()
print(f'Partidos en histórico: {len(matches):,}')

elo = EloRating().fit(matches).snapshot()
pi = PIRating().fit(matches).snapshot()
form = FormRating().fit(matches).snapshot()
print(f'Equipos rated por Elo: {len(elo)}')
print(f'Equipos rated por PI:  {len(pi)}')
print(f'Equipos rated por Form:{len(form)}')

## Top 20 por Elo

In [ ]:
print(elo.head(20).to_string(index=False))

## Distribuciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(elo['elo'], bins=30, color='#1f77b4', edgecolor='black')
axes[0].set_title('Distribución Elo')
axes[0].set_xlabel('Elo')
axes[0].axvline(1500, color='red', linestyle='--', label='base=1500')
axes[0].legend()

axes[1].hist(pi['pi_combined'], bins=30, color='#ff7f0e', edgecolor='black')
axes[1].set_title('Distribución PI combinada')
axes[1].set_xlabel('PI combined')

axes[2].hist(form['form_score'], bins=30, color='#2ca02c', edgecolor='black')
axes[2].set_title('Distribución Form score')
axes[2].set_xlabel('Form score')

plt.tight_layout()
plt.show()

## Rating compuesto (z-scored, ponderado)

Pesos default: Elo 0.55, PI 0.30, Form 0.15. Es la base que usa el simulador para la cache de probabilidades.

In [ ]:
ensemble = RatingEnsemble().fit(matches)
composite = ensemble.composite_table()
print(f'Equipos en composite: {len(composite)}')
print()
print('Top 25 por composite_strength:')
print(composite[['team', 'elo', 'pi_combined', 'form_score', 'composite_strength']].head(25).to_string(index=False))

## Correlaciones entre rating systems

In [ ]:
corr = composite[['elo', 'pi_combined', 'form_score', 'composite_strength']].corr()
print(corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='black')
fig.colorbar(im)
ax.set_title('Correlación entre ratings')
plt.tight_layout()
plt.show()

## Top equipos del Mundial 2026 por composite_strength

In [ ]:
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
wc = matches[(matches['competition'] == 'WC') & (matches['date'].dt.year == 2026)]
wc_teams = sorted(set(wc['team_a'].dropna()).union(wc['team_b'].dropna()))
print(f'Equipos en WC 2026: {len(wc_teams)}')

wc_strength = composite[composite['team'].isin(wc_teams)].copy()
print()
print('Top 15 Mundial 2026:')
print(wc_strength.head(15)[['team', 'elo', 'pi_combined', 'composite_strength']].to_string(index=False))
print()
print('Bottom 10 Mundial 2026 (probables underdogs):')
print(wc_strength.tail(10)[['team', 'elo', 'pi_combined', 'composite_strength']].to_string(index=False))

## Distribución de fuerza dentro del Mundial 2026

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
wc_strength_sorted = wc_strength.sort_values('composite_strength', ascending=True)
ax.barh(wc_strength_sorted['team'], wc_strength_sorted['composite_strength'], color='#1f77b4')
ax.set_xlabel('Composite strength (z-scored)')
ax.set_title('Fuerza compuesta de los 48 equipos del Mundial 2026')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()